<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Tutor%C3%ADa%205_Simulaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MODELO TOYCO: ORIGINAL VS ESCENARIOS DE SENSIBILIDAD
# CON GRÁFICAS EN PYTHON
# ============================================================

!pip install pulp

from pulp import *
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. FUNCIÓN PARA RESOLVER EL MODELO TOYCO
# ============================================================

def resolver_toyco(rhs1, rhs2, rhs3, nombre_escenario):
    modelo = LpProblem(nombre_escenario, LpMaximize)

    x1 = LpVariable("x1_Trenes", lowBound=0)
    x2 = LpVariable("x2_Camiones", lowBound=0)
    x3 = LpVariable("x3_Automoviles", lowBound=0)

    modelo += 3*x1 + 2*x2 + 5*x3, "Utilidad_total"

    modelo += x1 + 2*x2 + x3 <= rhs1, "Operacion_1"
    modelo += 3*x1 + 2*x3 <= rhs2, "Operacion_2"
    modelo += x1 + 4*x2 <= rhs3, "Operacion_3"

    modelo.solve(PULP_CBC_CMD(msg=False))

    return {
        "Escenario": nombre_escenario,
        "x1_Trenes": x1.varValue,
        "x2_Camiones": x2.varValue,
        "x3_Automoviles": x3.varValue,
        "Z_Utilidad": value(modelo.objective),
        "Holgura_Op1": modelo.constraints["Operacion_1"].slack,
        "Holgura_Op2": modelo.constraints["Operacion_2"].slack,
        "Holgura_Op3": modelo.constraints["Operacion_3"].slack,
        "Precio_Sombra_Op1": modelo.constraints["Operacion_1"].pi,
        "Precio_Sombra_Op2": modelo.constraints["Operacion_2"].pi,
        "Precio_Sombra_Op3": modelo.constraints["Operacion_3"].pi,
        "Costo_Reducido_x1": x1.dj,
        "Costo_Reducido_x2": x2.dj,
        "Costo_Reducido_x3": x3.dj
    }

# ============================================================
# 2. RESOLVER MODELO ORIGINAL Y ESCENARIOS
# ============================================================

resultados = []

resultados.append(resolver_toyco(430, 460, 420, "Original"))
resultados.append(resolver_toyco(430, 470, 420, "Op. 2 +10 min"))
resultados.append(resolver_toyco(430, 460, 435, "Op. 3 +15 min"))
resultados.append(resolver_toyco(410, 460, 420, "Op. 1 -20 min"))

datos = pd.DataFrame(resultados)

# Cambio en Z frente al modelo original
datos["Cambio_Z"] = datos["Z_Utilidad"] - datos.loc[0, "Z_Utilidad"]

print("TABLA DE RESULTADOS")
display(datos)

# ============================================================
# 3. EXPLICACIÓN DE RESULTADOS EN TEXTO
# ============================================================

print("\nINTERPRETACIÓN GENERAL")
print("El modelo original tiene Z =", datos.loc[0, "Z_Utilidad"])
print("La operación 2 tiene el mayor precio sombra en el modelo original.")
print("Esto significa que aumentar minutos en la operación 2 genera mayor impacto en la utilidad.")
print("La operación 3 tiene holgura, por eso aumentar su capacidad no mejora Z.")

# ============================================================
# 4. GRÁFICA 1: COMPARACIÓN DE UTILIDAD Z
# ============================================================

colores = ["#4C78A8", "#54A24B", "#F58518", "#E45756"]

plt.figure(figsize=(10, 6))
plt.bar(datos["Escenario"], datos["Z_Utilidad"], color=colores)

plt.title("Comparación de utilidad: modelo original vs escenarios", fontsize=14)
plt.xlabel("Escenario")
plt.ylabel("Utilidad máxima Z")
plt.xticks(rotation=20)

for i, valor in enumerate(datos["Z_Utilidad"]):
    plt.text(i, valor + 5, str(round(valor, 2)), ha="center", fontsize=11)

plt.show()

# ============================================================
# 5. GRÁFICA 2: CAMBIO EN LA UTILIDAD
# ============================================================

plt.figure(figsize=(10, 6))
plt.bar(datos["Escenario"], datos["Cambio_Z"], color=colores)

plt.axhline(0, color="black", linewidth=1)

plt.title("Cambio en la utilidad frente al modelo original", fontsize=14)
plt.xlabel("Escenario")
plt.ylabel("Cambio en Z")
plt.xticks(rotation=20)

for i, valor in enumerate(datos["Cambio_Z"]):
    posicion = valor + 1 if valor >= 0 else valor - 3
    plt.text(i, posicion, str(round(valor, 2)), ha="center", fontsize=11)

plt.show()

# ============================================================
# 6. GRÁFICA 3: MEZCLA ÓPTIMA DE PRODUCCIÓN
# ============================================================

plt.figure(figsize=(11, 6))

ancho = 0.25
posiciones = range(len(datos["Escenario"]))

plt.bar(
    [p - ancho for p in posiciones],
    datos["x1_Trenes"],
    width=ancho,
    label="Trenes",
    color="#4C78A8"
)

plt.bar(
    posiciones,
    datos["x2_Camiones"],
    width=ancho,
    label="Camiones",
    color="#F58518"
)

plt.bar(
    [p + ancho for p in posiciones],
    datos["x3_Automoviles"],
    width=ancho,
    label="Automóviles",
    color="#54A24B"
)

plt.title("Mezcla óptima de producción por escenario", fontsize=14)
plt.xlabel("Escenario")
plt.ylabel("Cantidad producida")
plt.xticks(posiciones, datos["Escenario"], rotation=20)
plt.legend()

plt.show()

# ============================================================
# 7. GRÁFICA 4: PRECIOS SOMBRA DEL MODELO ORIGINAL
# ============================================================

operaciones = ["Operación 1", "Operación 2", "Operación 3"]

precios_sombra_original = [
    datos.loc[0, "Precio_Sombra_Op1"],
    datos.loc[0, "Precio_Sombra_Op2"],
    datos.loc[0, "Precio_Sombra_Op3"]
]

plt.figure(figsize=(8, 5))
plt.bar(
    operaciones,
    precios_sombra_original,
    color=["#F58518", "#54A24B", "#E45756"]
)

plt.title("Precios sombra del modelo original", fontsize=14)
plt.xlabel("Operación")
plt.ylabel("Precio sombra")

for i, valor in enumerate(precios_sombra_original):
    plt.text(i, valor + 0.05, str(round(valor, 2)), ha="center", fontsize=11)

plt.show()

# ============================================================
# 8. GRÁFICA 5: HOLGURAS DEL MODELO ORIGINAL
# ============================================================

holguras_original = [
    datos.loc[0, "Holgura_Op1"],
    datos.loc[0, "Holgura_Op2"],
    datos.loc[0, "Holgura_Op3"]
]

plt.figure(figsize=(8, 5))
plt.bar(
    operaciones,
    holguras_original,
    color=["#E45756", "#E45756", "#54A24B"]
)

plt.title("Holguras del modelo original", fontsize=14)
plt.xlabel("Operación")
plt.ylabel("Minutos sobrantes")

for i, valor in enumerate(holguras_original):
    plt.text(i, valor + 0.5, str(round(valor, 2)), ha="center", fontsize=11)

plt.show()

# ============================================================
# 9. GRÁFICA 6: HOLGURA VS PRECIO SOMBRA
# ============================================================

decision = pd.DataFrame({
    "Operación": operaciones,
    "Precio sombra": precios_sombra_original,
    "Holgura": holguras_original
})

plt.figure(figsize=(9, 6))

plt.scatter(
    decision["Holgura"],
    decision["Precio sombra"],
    s=300,
    color=["#F58518", "#54A24B", "#E45756"]
)

for i in range(len(decision)):
    plt.text(
        decision["Holgura"][i] + 0.3,
        decision["Precio sombra"][i],
        decision["Operación"][i],
        fontsize=11
    )

plt.title("Relación entre holgura y precio sombra", fontsize=14)
plt.xlabel("Holgura")
plt.ylabel("Precio sombra")
plt.grid(True, alpha=0.3)

plt.show()

# ============================================================
# 10. GRÁFICA 7: COSTOS REDUCIDOS DEL MODELO ORIGINAL
# ============================================================

variables = ["x1 Trenes", "x2 Camiones", "x3 Automóviles"]

costos_reducidos_original = [
    datos.loc[0, "Costo_Reducido_x1"],
    datos.loc[0, "Costo_Reducido_x2"],
    datos.loc[0, "Costo_Reducido_x3"]
]

plt.figure(figsize=(8, 5))
plt.bar(
    variables,
    costos_reducidos_original,
    color=["#E45756", "#54A24B", "#54A24B"]
)

plt.axhline(0, color="black", linewidth=1)

plt.title("Costos reducidos del modelo original", fontsize=14)
plt.xlabel("Variable")
plt.ylabel("Costo reducido")

for i, valor in enumerate(costos_reducidos_original):
    plt.text(i, valor + 0.2, str(round(valor, 2)), ha="center", fontsize=11)

plt.show()

# ============================================================
# 11. DEMOSTRACIÓN DE PRECIOS SOMBRA CON +1 UNIDAD
# ============================================================

z_original = datos.loc[0, "Z_Utilidad"]

z_op1_mas_1 = resolver_toyco(431, 460, 420, "Op. 1 +1 min")["Z_Utilidad"]
z_op2_mas_1 = resolver_toyco(430, 461, 420, "Op. 2 +1 min")["Z_Utilidad"]
z_op3_mas_1 = resolver_toyco(430, 460, 421, "Op. 3 +1 min")["Z_Utilidad"]

demostracion = pd.DataFrame({
    "Operación": operaciones,
    "Z original": [z_original, z_original, z_original],
    "Z con +1 unidad": [z_op1_mas_1, z_op2_mas_1, z_op3_mas_1],
    "Cambio en Z": [
        z_op1_mas_1 - z_original,
        z_op2_mas_1 - z_original,
        z_op3_mas_1 - z_original
    ]
})

print("\nDEMOSTRACIÓN DE DÓNDE SALEN LOS PRECIOS SOMBRA")
display(demostracion)

# ============================================================
# 12. GRÁFICA 8: DEMOSTRACIÓN DEL CAMBIO EN Z POR +1 UNIDAD
# ============================================================

plt.figure(figsize=(8, 5))
plt.bar(
    demostracion["Operación"],
    demostracion["Cambio en Z"],
    color=["#F58518", "#54A24B", "#E45756"]
)

plt.title("Cambio en Z al aumentar 1 unidad de recurso", fontsize=14)
plt.xlabel("Operación")
plt.ylabel("Cambio en Z")

for i, valor in enumerate(demostracion["Cambio en Z"]):
    plt.text(i, valor + 0.05, str(round(valor, 2)), ha="center", fontsize=11)

plt.show()

# ============================================================
# 13. CONCLUSIONES AUTOMÁTICAS
# ============================================================

print("\nCONCLUSIONES")
print("1. La solución original recomienda producir:")
print("   Trenes:", datos.loc[0, "x1_Trenes"])
print("   Camiones:", datos.loc[0, "x2_Camiones"])
print("   Automóviles:", datos.loc[0, "x3_Automoviles"])
print("   Utilidad máxima:", datos.loc[0, "Z_Utilidad"])

print("\n2. La operación 2 es el recurso más valioso porque tiene el mayor precio sombra.")
print("3. La operación 3 tiene holgura, por eso aumentar su capacidad no mejora la utilidad.")
print("4. El escenario Op. 2 +10 min mejora la utilidad en:", datos.loc[1, "Cambio_Z"])
print("5. El escenario Op. 1 -20 min reduce la utilidad en:", datos.loc[3, "Cambio_Z"])